# Excerpt of dualisation workflow

Auto dualisation of LPs (WIP) including plotting routine

In [ ]:
import logging
import matplotlib.pyplot as plt
import numpy as np
import pypsa
from linopy import Model

logger = logging.getLogger(__name__)
plt.rcParams["figure.dpi"] = 90

solver_name = "gurobi"
solver_options = {
    "OutputFlag": 0,
    "FeasibilityTol": 1e-8,
    "OptimalityTol": 1e-8,
    "Method": 2,  # Barrier method,
    "Crossover": 1,
}

solver_options_debug = {
    "OutputFlag": 1,
    "FeasibilityTol": 1e-8,
    "OptimalityTol": 1e-8,
    "Method": 2,  # Barrier method,
    "InfUnbdInfo": 1,
    "DualReductions": 0,
}

from tqdm.notebook import tqdm

In [ ]:
def add_battery_constraints(n):
    """
    Add constraint ensuring that charger = discharger, i.e.
    1 * charger_size - efficiency * discharger_size = 0
    """
    if not n.links.p_nom_extendable.any():
        return

    discharger_bool = n.links.index.str.contains("battery discharger")
    charger_bool = n.links.index.str.contains("battery charger")

    dischargers_ext = n.links[discharger_bool].query("p_nom_extendable").index
    chargers_ext = n.links[charger_bool].query("p_nom_extendable").index

    eff = n.links.efficiency[dischargers_ext].values
    lhs = (
        n.model["Link-p_nom"].loc[chargers_ext]
        - n.model["Link-p_nom"].loc[dischargers_ext] * eff
    )

    n.model.add_constraints(lhs == 0, name="Link-charger_ratio")

In [ ]:
n = pypsa.Network("50-node-network.nc")

Fix dual degeneracy by adding costs

In [ ]:
n.storage_units["marginal_cost_storage"] = 1e-4
n.storage_units["spill_cost"] = 1e-4

In [ ]:
n.optimize.create_model(include_objective_constant=False)
add_battery_constraints(n)

Convert bounds to constraints

In [ ]:
def bounds_to_constraints(m1: Model) -> None:
    """Add explicit bound constraints for variables with bounds set directly
    in the variable rather than via explicit constraints.
    
    Adds constraints named '{var_name}-bound-lower' and '{var_name}-bound-upper'
    to distinguish from PyPSA's automatic '-fix-*' constraints.
    
    Also resets variable bounds to [-inf, inf] after adding constraints,
    to avoid double-counting in the dual.
    """
    for var_name, var in m1.variables.items():
        mask = var.labels != -1
        lb = var.lower
        ub = var.upper
        
        # lower bound
        if f"{var_name}-bound-lower" not in m1.constraints:
            has_finite_lb = np.isfinite(lb.values[mask.values]).any()
            if has_finite_lb:
                m1.add_constraints(
                    var >= lb,
                    name=f"{var_name}-bound-lower",
                    mask=mask,
                )
                logger.info(f"Added lower bound constraint for '{var_name}'.")
                var.lower.values[mask.values] = -np.inf
                # Remove bounds to avoid double-counting in the dual. Rely on the new constraints instead.
                m1.variables[var_name].lower.values[mask.values] = -np.inf
        
        # upper bound
        if f"{var_name}-bound-upper" not in m1.constraints:
            has_finite_ub = np.isfinite(ub.values[mask.values]).any()
            if has_finite_ub:
                m1.add_constraints(
                    var <= ub,
                    name=f"{var_name}-bound-upper",
                    mask=mask,
                )
                logger.info(f"Added upper bound constraint for '{var_name}'.")
                var.upper.values[mask.values] = np.inf
                # Remove bounds to avoid double-counting in the dual. Rely on the new constraints instead.
                m1.variables[var_name].upper.values[mask.values] = np.inf

In [ ]:
m1 = n.model

In [ ]:
bounds_to_constraints(m1)

### Visualise constraint matrix sparsity

In [ ]:
A_k = m1.constraints.to_matrix()

# variable column blocks
var_blocks = {}
offset = 0
for var_name, var in m1.variables.items():
    var_blocks[var_name] = (offset, offset + var.size)
    offset += var.size

# constraint row blocks
con_blocks = {}
offset = 0
for con_name, con in m1.constraints.items():
    con_blocks[con_name] = (offset, offset + con.size)
    offset += con.size

n_rows = A_k.shape[0]
n_cols = A_k.shape[1]
colors = plt.cm.plasma(np.linspace(0, 1, len(var_blocks)))

fig, ax = plt.subplots(figsize=(8, 6))

cx = A_k.tocsc()
for (var_name, (col_start, col_end)), color in zip(var_blocks.items(), colors):
    block = cx[:, col_start:col_end].tocoo()
    ax.scatter(block.col + col_start, block.row, color=color, s=0.5, label=var_name)

# set main axis limits first
ax.set_xlim(0, n_cols)
ax.set_ylim(n_rows, 0)
ax.set_xlabel("Variables")
ax.set_ylabel("Constraints")
ax.set_title("Constraint matrix sparsity by variable")
ax.legend(loc="upper right", markerscale=5)

# horizontal separators
for con_name, (row_start, _) in con_blocks.items():
    ax.axhline(row_start, color="gray", linewidth=0.4, linestyle="--", alpha=0.6)

# right y-axis labels — set limits explicitly to match
con_mids = [(row_start + row_end) / 2 for _, (row_start, row_end) in con_blocks.items()]
con_names = list(con_blocks.keys())

ax2 = ax.twinx()
ax2.set_ylim(n_rows, 0)  # explicitly match after main axis is set
ax2.set_yticks(con_mids)
ax2.set_yticklabels(con_names, fontsize=7)

plt.tight_layout()
plt.show()

### Construct dual

In [ ]:
m2 = Model()

Dual variables including non-negativity bounds

In [ ]:
dual_vars = {}
for name, con in tqdm(m1.constraints.items(), desc="Adding dual variables", total=len(m1.constraints)):
    sign_vals = con.sign.values.flatten()
    if len(sign_vals) == 0:
        logger.info(f"Skipping constraint '{name}' — no active rows.")
        continue
    
    is_equality = sign_vals[0] == "="
    var_type = "free" if is_equality else "non-negative"
    
    mask = con.labels != -1
    if not mask.any():
        logger.info(f"Skipping constraint '{name}' — all rows masked.")
        continue
    
    logger.info(f"Adding {var_type} dual variable for constraint '{name}' with shape {con.shape} and dims {con.labels.dims}.")
    dual_vars[name] = m2.add_variables(
        lower=0 if not is_equality else -np.inf,
        upper=np.inf,
        coords=list(con.coords.values()),
        name=name,
        mask=mask,
    )

Create reverse lookup: flat variable index -> (variable name, coordinate dict)

In [ ]:
# TODOMAKEROBUST

var_lookup = {}
for var_name, var in tqdm(m1.variables.items(), desc="Building variable label lookup"):
    labels = var.labels
    logger.info(f"Creating label lookup for variable '{var_name}' with shape {labels.shape} and dims {labels.dims}.")
    
    flat_labels = labels.values.flatten()
    coord_arrays = np.meshgrid(
        *[labels.coords[dim].values for dim in labels.dims],
        indexing='ij'
    )
    flat_coords = [arr.flatten() for arr in coord_arrays]
    
    for k, flat in enumerate(flat_labels):
        if flat != -1:
            var_lookup[int(flat)] = (
                var_name,
                {dim: flat_coords[i][k] for i, dim in enumerate(labels.dims)}
            )

Objective of dual problem

In [ ]:
dual_obj = 0
sense = "max" if m1.objective.sense == "min" else "min"
for name, con in m1.constraints.items():
    if name not in dual_vars:
        continue
    sign_vals = con.sign.values.flatten()
    if len(sign_vals) == 0:
        continue
    
    mask = con.labels != -1
    rhs_masked = con.rhs.where(mask, 0)
    sign = sign_vals[0]
    if sign == "<=":
        dual_obj -= (rhs_masked * dual_vars[name]).sum()
    else:
        dual_obj += (rhs_masked * dual_vars[name]).sum()

logger.info(f"Constructed dual objective with {len(dual_obj.coeffs)} terms.")
logger.info("Adding dual objective to model.")
m2.add_objective(dual_obj, sense=sense, overwrite=True)

Build dual feasibility constraints

In [ ]:
con_lookup = {}
for con_name, con in tqdm(m1.constraints.items(), desc="Building constraint label lookup"):
    labels = con.labels
    flat_labels = labels.values.flatten()
    
    if len(flat_labels) == 0:
        continue
    if not (flat_labels != -1).any():
        continue
    
    coord_arrays = np.meshgrid(
        *[labels.coords[dim].values for dim in labels.dims],
        indexing='ij'
    ) if len(labels.dims) > 0 else []
    flat_coords = [arr.flatten() for arr in coord_arrays]
    
    for k, flat in enumerate(flat_labels):
        if flat != -1:
            con_lookup[int(flat)] = (
                con_name,
                {dim: flat_coords[i][k] for i, dim in enumerate(labels.dims)}
            )

In [ ]:
sign_factors = {}
for con_name, con in m1.constraints.items():
    sign_vals = con.sign.values.flatten()
    if len(sign_vals) == 0:
        continue
    sign_factors[con_name] = -1 if sign_vals[0] == "<=" else 1

A_csc = m1.matrices.A.tocsc()
c = m1.matrices.c
indptr = A_csc.indptr
indices = A_csc.indices
data = A_csc.data
vlabels = m1.matrices.vlabels  # column index -> flat variable label
clabels = m1.matrices.clabels  # row index -> flat constraint label

dual_feas_terms = {var_name: {} for var_name in m1.variables}
for i in tqdm(range(A_csc.shape[1]), desc="Building dual feasibility terms"):
    flat_var = vlabels[i]
    if flat_var == -1:
        continue
    if flat_var not in var_lookup:
        continue
    var_name, var_coords = var_lookup[flat_var]
    terms = []
    for k in range(indptr[i], indptr[i+1]):
        j = indices[k]
        flat_con = clabels[j]
        if flat_con == -1:
            continue
        if flat_con not in con_lookup:
            continue
        if flat_con not in [con_lookup[flat_con][0]] and con_lookup[flat_con][0] not in sign_factors:
            continue
        coeff = data[k]
        con_name, con_coords = con_lookup[flat_con]
        if con_name not in sign_factors:
            continue
        terms.append((con_name, con_coords, sign_factors[con_name] * coeff))
    dual_feas_terms[var_name][flat_var] = (var_coords, terms, c[i])

In [ ]:
import pandas as pd
import xarray as xr

In [ ]:
from linopy.expressions import LinearExpression

In [ ]:
# build c lookup by flat variable label
c_by_label = {vlabels[i]: c[i] for i in range(len(vlabels))}

for var_name, var in tqdm(m1.variables.items(), desc="Adding dual feasibility constraints"):
    coords = [pd.Index(var.labels.coords[dim].values, name=dim) for dim in var.labels.dims]
    
    mask = var.labels != -1

    c_vals = xr.DataArray(
        np.vectorize(lambda flat: c_by_label.get(flat, 0.0))(var.labels.values),
        coords=var.labels.coords
    )

    def rule(m, *coord_vals, vname=var_name, vdims=var.labels.dims):
        coord_dict = dict(zip(vdims, coord_vals))
        flat = var.labels.sel(**coord_dict).item()
        if flat == -1:
            return None
        _, terms, _ = dual_feas_terms[vname][flat]
        return sum(
            coeff * dual_vars[con_name].at[tuple(con_coords.values())]
            for con_name, con_coords, coeff in terms
        )

    lhs = LinearExpression.from_rule(m2, rule, coords)
    m2.add_constraints(lhs == c_vals, name=var_name, mask=mask)

In [ ]:
def plot_sparsity(ax, matrix, var_blocks, con_blocks, title, colors):
    n_rows = matrix.shape[0]
    n_cols = matrix.shape[1]
    cx = matrix.tocsc()
    for (var_name, (col_start, col_end)), color in zip(var_blocks.items(), colors):
        block = cx[:, col_start:col_end].tocoo()
        ax.scatter(block.col + col_start, block.row, color=color, s=0.5, label=var_name)
    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)
    ax.set_xlabel("Variables")
    ax.set_ylabel("Constraints")
    ax.set_title(title)
    # vertical separators for variable blocks
    for var_name, (col_start, col_end) in var_blocks.items():
        ax.axvline(col_start, color="gray", linewidth=0.4, linestyle="--", alpha=0.6)
    # horizontal separators for constraint blocks
    for con_name, (row_start, _) in con_blocks.items():
        ax.axhline(row_start, color="gray", linewidth=0.4, linestyle="--", alpha=0.6)
    # right y-axis labels
    con_mids = [(row_start + row_end) / 2 for _, (row_start, row_end) in con_blocks.items()]
    con_names = list(con_blocks.keys())
    ax2 = ax.twinx()
    ax2.set_ylim(n_rows, 0)
    ax2.set_yticks(con_mids)
    ax2.set_yticklabels(con_names, fontsize=7)
    return ax


def get_blocks(model_vars, model_cons):
    var_blocks = {}
    offset = 0
    for var_name, var in model_vars.items():
        var_blocks[var_name] = (offset, offset + var.size)
        offset += var.size
    con_blocks = {}
    offset = 0
    for con_name, con in model_cons.items():
        con_blocks[con_name] = (offset, offset + con.size)
        offset += con.size
    return var_blocks, con_blocks


primal_var_blocks, primal_con_blocks = get_blocks(m1.variables, m1.constraints)
dual_var_blocks, dual_con_blocks = get_blocks(m2.variables, m2.constraints)

colors_primal = plt.cm.tab20(np.linspace(0, 1, len(primal_var_blocks)))
colors_dual = plt.cm.tab20(np.linspace(0, 1, len(dual_var_blocks)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7.5))

plot_sparsity(ax1, m1.constraints.to_matrix(), primal_var_blocks, primal_con_blocks, "Primal constraint matrix", colors_primal)
plot_sparsity(ax2, m2.constraints.to_matrix(), dual_var_blocks, dual_con_blocks, "Dual constraint matrix", colors_dual)

handles1 = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=6, label=n)
            for n, c in zip(primal_var_blocks.keys(), colors_primal)]
handles2 = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=6, label=n)
            for n, c in zip(dual_var_blocks.keys(), colors_dual)]

# legends anchored below each subplot, aligned top-right
ax1.legend(handles=handles1, title="Primal variables", fontsize=8,
           loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
ax2.legend(handles=handles2, title="Dual variables", fontsize=8,
           loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3)

plt.tight_layout()
plt.show()

In [ ]:
# Export to png
fig.savefig("fig/constraint_matrices.png", dpi=300, bbox_inches="tight")

In [ ]:
m1.solve(solver_name=solver_name, **solver_options_debug)

In [ ]:
violations = {}

for var_name, var in m1.variables.items():
    max_violation = 0.0
    for idx in np.ndindex(var.labels.shape):
        flat = var.labels.values[idx]
        if flat == -1:
            continue
        var_coords, terms, obj_coeff = dual_feas_terms[var_name][flat]
        
        lhs_val = 0.0
        for con_name, con_coords, coeff in terms:
            sign = m1.constraints[con_name].sign.values.flatten()[0]
            dual_val = m1.constraints[con_name].dual.sel(**con_coords).item()
            # linopy negates <= duals, flip back to our convention
            if sign == "<=":
                dual_val = -dual_val
            lhs_val += coeff * dual_val
        
        violation = abs(lhs_val - obj_coeff)
        max_violation = max(max_violation, violation)
    
    violations[var_name] = max_violation
    print(f"{var_name}: max violation = {max_violation:.2e}")

In [ ]:
m2.solve(solver_name=solver_name, **solver_options_debug)

In [ ]:
print(m2.status, m2.termination_condition)
print("Dual objective:", m2.objective.value)
print("Primal objective:", m1.objective.value)
print("Gap:", abs(m2.objective.value - m1.objective.value))
print("Relative gap:", abs(m2.objective.value - m1.objective.value) / abs(m1.objective.value))

In [ ]:
m1.variables

In [ ]:
var_name = "StorageUnit-state_of_charge"

primal = m1.variables[var_name].solution.to_dataframe().squeeze()
print(primal.head())
dual = m2.constraints[var_name].dual.to_dataframe().squeeze()
print(dual.head())

both = primal.copy()
both = pd.DataFrame({
    "primal": primal,
    "dual": dual,
})

# Plot scatter primal and dual
both.plot.scatter(x="primal", y="dual")

# Export to png
fig.savefig("fig/primal_dual_scatter_soc.png", dpi=300, bbox_inches="tight")
